# 3. Inspecting Your Reference Molecules

Before augmentation and fine-tuning, explore the molecules in your training
split to understand the chemical space you are targeting.

**What you will do:**
1. Load your training split and check basic quality (validity, uniqueness)
2. Visualise a sample of the structures
3. Compute and plot physicochemical property distributions
4. Explore scaffold diversity
5. Compute a full metrics summary
6. Use `compare_distributions` to formally test distribution differences

**Input:** Load your training split from `02_data_splitting.ipynb`.
Point `SMILES` to the file saved by that notebook
(e.g. `load_smiles("dataset/train.csv")`).

Mathematical definitions for all metrics are in `docs/metrics_reference.md`.

## Setup

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

from evaluation import load_smiles
from evaluation.metrics import (
    validity,
    uniqueness,
    mean_pairwise_distance,
    scaffold_entropy,
    nearest_neighbor_distance,
    fraction_passing_lipinski,
)
from evaluation.properties import compute_properties
from evaluation.visualization import (
    draw_molecule_grid,
    plot_distribution,
    plot_property_panel,
    plot_scaffold_frequencies,
    plot_distribution_comparison,
    compare_distributions,
)

print("Imports OK")

In [ ]:
# Set the base directory to the current working directory (where this notebook is located)
base_dir = os.path.dirname(os.path.abspath("__file__"))

## Load Molecules

Replace the path below with your own file.

**Supported file formats:**
- `.smi` / `.txt` — one SMILES per line (first token is the SMILES, the rest is ignored)
- `.csv` — must contain a column named `smiles` (case-insensitive)

In [ ]:
# Load your training split SMILES from file
smiles_path = os.path.join(base_dir, "your_path_here")

SMILES = load_smiles(smiles_path)

print(f"Loaded {len(SMILES)} SMILES strings.")

---
## Step 1 — Inspect

Before computing any metrics, look at your molecules and check basic quality.

In [ ]:
# ── Basic quality statistics ─────────────────────────────────────────────────
v = validity(SMILES)
u = uniqueness(SMILES)

print(f"Total SMILES      : {len(SMILES)}")
print(f"Validity          : {v:.3f}  ({int(v * len(SMILES))} valid)")
print(f"Uniqueness        : {u:.3f}  ({int(u * v * len(SMILES))} unique valid)")

In [ ]:
# ── Draw a grid of the first 20 molecules ────────────────────────────────────
# The function accepts a list of SMILES or RDKit Mol objects.
image = draw_molecule_grid(SMILES[:20], n_cols=5)
display(image)

---
## Step 2 — Molecular Property Distributions

`compute_properties` returns a DataFrame with one row per molecule and one
column per property.  We can then plot all distributions at once with
`plot_property_panel`.

In [ ]:
# ── Compute all properties ────────────────────────────────────────────────────
properties = compute_properties(SMILES)
properties.head()

In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────────
properties.describe().round(2)

In [ ]:
# ── Plot all property distributions ──────────────────────────────────────────
fig = plot_property_panel(properties)
plt.show()

In [ ]:
# ── Drug-likeness: what fraction passes Lipinski's Ro5? ───────────────────────
lipinski_fraction = fraction_passing_lipinski(SMILES)
print(f"Fraction passing Lipinski's Ro5: {lipinski_fraction:.3f}")

In [ ]:
# ── Scaffold overview ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
plot_scaffold_frequencies(SMILES, top_n=10, ax=ax)
plt.show()

In [ ]:
metrics_summary = {
    "validity"                  : validity(SMILES),
    "uniqueness"                : uniqueness(SMILES),
    "mean_pairwise_distance"    : mean_pairwise_distance(SMILES),
    "scaffold_entropy"          : scaffold_entropy(SMILES),
    "nearest_neighbor_distance" : nearest_neighbor_distance(SMILES),
    "fraction_passing_lipinski" : fraction_passing_lipinski(SMILES),
}

print("Metric summary")
print("-" * 44)
for name, value in metrics_summary.items():
    print(f"  {name:<40} {value:.4f}")

---
## Step 4 — Statistical Comparison

`compare_distributions` tests whether two sets of molecules have significantly
different property distributions (Kolmogorov-Smirnov test by default).

This is the same function used in notebook 06 to compare generated molecules
against your reference sets.  Here we preview it by splitting the molecule
list in half and comparing the two halves — they should be similar.

In [ ]:
set_a = SMILES[:len(SMILES) // 2]
set_b = SMILES[len(SMILES) // 2:]

props_a = compute_properties(set_a)
props_b = compute_properties(set_b)

results = []
for prop in ["molecular_weight", "logp", "topological_polar_surface_area"]:
    result = compare_distributions(props_a[prop], props_b[prop], test="ks")
    results.append({"property": prop, "statistic": round(result["statistic"], 3),
                    "p_value": round(result["p_value"], 3), "significant": result["significant"]})

pd.DataFrame(results)

---
## Student Challenges

---

### Challenge 1 — Compare two subsets

Split your molecule list in half (by index) and treat them as two separate
sets.  Use `mean_pairwise_distance` and `nearest_neighbor_distance` to measure
how structurally different they are from each other.

- Are the two halves more or less diverse than the full set?
- Is the nearest-neighbour distance between them larger or smaller than the
  internal distance within each half?

The code cell below gets you started — extend it with the metrics above.

### Challenge 2 — Compare two molecular sets

You will repeat this notebook in notebook 06, comparing generated molecules
against the fine-tuning set.  Here, practise with two arbitrary sets of your
choice — for example, the training split vs a random sample from the full dataset.

Use `compare_distributions` and `plot_distribution_comparison` to:
1. Test whether their property distributions differ significantly.
2. Compute novelty of one set relative to the other.
3. Compute the nearest-neighbour distance from one set to the other.

This gives you a baseline understanding of what "similar" and "different"
look like before you evaluate a model's output.

In [ ]:
set_a = SMILES[:len(SMILES) // 2]
set_b = SMILES[len(SMILES) // 2:]

# Your analysis here — compute metrics on set_a, set_b, and between them